In [ ]:
# ============================================================
# CELL 1: Install Dependencies
# Install dev version of transformers (required for Qwen3.5-0.8B
# model type 'qwen3_5' not available in stable release),
# huggingface_hub, accelerate, and datasets.
# After this cell runs, the kernel must be restarted.
# ============================================================
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/huggingface/transformers.git",
    "huggingface_hub>=0.23.0",
    "accelerate>=0.26.0",
    "datasets>=2.18.0",
])
print("Install complete — restart kernel now")

In [ ]:
# ============================================================
# CELL 2: Imports, Global Constants, and Data Loading (Phase 1)
#
# - Loads the ailsntua/QEvasion dataset from HuggingFace Hub
# - Renames columns to: id, question, answer, label
# - Creates stratified 80/20 train/val split (SEED=42)
# - Computes class weights for weighted CrossEntropyLoss
#   (needed because dataset is imbalanced:
#    Ambivalent ~59%, Clear Reply ~30%, Clear Non-Reply ~10%)
# - Verifies test set length matches sample_solution.csv (308 rows)
#
# LABEL2ID uses alphabetical ordering:
#   Ambivalent=0, Clear Non-Reply=1, Clear Reply=2
# ============================================================
import os, random, gc, re, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          AutoModelForCausalLM, get_linear_schedule_with_warmup)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, f1_score,
                             precision_recall_fscore_support,
                             classification_report, confusion_matrix)
from tqdm.auto import tqdm
import transformers
print(f"Transformers: {transformers.__version__}")

SEED          = 42
LABELS        = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]
LABEL2ID      = {l: i for i, l in enumerate(sorted(LABELS))}
ID2LABEL      = {i: l for l, i in LABEL2ID.items()}
LABEL_COL     = "label"
QUESTION_COL  = "question"
ANSWER_COL    = "answer"
VALID_LABELS  = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]
FALLBACK_LABEL= "Ambivalent"
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
print(f"Device: {DEVICE}")

ds = load_dataset("ailsntua/QEvasion")
train_df = ds['train'].to_pandas()
test_df  = ds['test'].to_pandas()

train_df = train_df[['index','interview_question','interview_answer','clarity_label']].copy()
test_df  = test_df[['index','interview_question','interview_answer','clarity_label']].copy()
train_df.columns = ['id','question','answer','label']
test_df.columns  = ['id','question','answer','label']

sample_sub = pd.read_csv('/kaggle/input/competitions/ys-19-2025-2026-assignment-4/sample_solution.csv')
assert len(test_df) == len(sample_sub)

train_df["label_id"] = train_df[LABEL_COL].map(LABEL2ID)
test_df["label_id"]  = test_df[LABEL_COL].map(LABEL2ID)

train_split, val_split = train_test_split(
    train_df, test_size=0.20, random_state=SEED,
    stratify=train_df["label_id"])
train_split = train_split.reset_index(drop=True)
val_split   = val_split.reset_index(drop=True)
test_df     = test_df.reset_index(drop=True)

class_weights = compute_class_weight(
    "balanced", classes=np.arange(3), y=train_split["label_id"].values)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

print(f"Train: {len(train_split)} | Val: {len(val_split)} | Test: {len(test_df)}")
print("Train label distribution:", Counter(train_df['label']))
print("--- Phase 1 Complete ---")

In [ ]:
# ============================================================
# CELL 3: Encoder Baselines — BERT and DistilBERT (Phase 2A)
#
# Implements the ClarityDataset (tokenizes question+answer pairs)
# and a generic train_encoder() function used for all encoder models.
#
# Training setup (matches HW2 baselines):
#   - AdamW optimizer, no weight decay on bias/LayerNorm
#   - Linear warmup schedule (10% of total steps)
#   - Weighted CrossEntropyLoss (handles class imbalance)
#   - Early stopping on validation Macro F1
#   - Gradient accumulation for effective larger batch sizes
#
# BERT:      max_len=512, bs=16, ga=1, epochs=8, lr=2e-5, patience=3
# DistilBERT: max_len=512, bs=16, ga=1, epochs=8, lr=1e-5, patience=3
#
# Results stored in encoder_results, encoder_val_preds,
# encoder_val_true, encoder_test_preds dictionaries.
# ============================================================
class ClarityDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length, is_test=False):
        self.data      = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length= max_length
        self.is_test   = is_test
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        enc = self.tokenizer(
            str(row[QUESTION_COL]), str(row[ANSWER_COL]),
            max_length=self.max_length, padding="max_length",
            truncation=True, return_tensors="pt")
        item = {"input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0)}
        if "token_type_ids" in enc:
            item["token_type_ids"] = enc["token_type_ids"].squeeze(0)
        if not self.is_test:
            item["labels"] = torch.tensor(row["label_id"], dtype=torch.long)
        return item

def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss, all_preds, all_true = 0.0, [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            lbls = batch["labels"].to(device)
            kwargs = {"input_ids": ids, "attention_mask": mask}
            if "token_type_ids" in batch:
                kwargs["token_type_ids"] = batch["token_type_ids"].to(device)
            logits = model(**kwargs).logits
            total_loss += loss_fn(logits, lbls).item()
            all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            all_true.extend(lbls.cpu().numpy())
    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_true, all_preds)
    macro_f1 = f1_score(all_true, all_preds, average="macro", zero_division=0)
    f1w      = f1_score(all_true, all_preds, average="weighted", zero_division=0)
    return avg_loss, acc, f1w, macro_f1, 0, 0, all_preds, all_true

def train_encoder(model_name, max_length, batch_size, grad_accum,
                  num_epochs, lr, patience, force_float32=False):
    print(f"\n{'='*60}\nTraining: {model_name}\n{'='*60}")
    set_seed()
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    tr_ds = ClarityDataset(train_split, tokenizer, max_length)
    va_ds = ClarityDataset(val_split,   tokenizer, max_length)
    te_ds = ClarityDataset(test_df,     tokenizer, max_length, is_test=True)
    tr_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,  num_workers=2)
    va_loader = DataLoader(va_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    te_loader = DataLoader(te_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    gc.collect(); torch.cuda.empty_cache()
    kwargs = {"num_labels":3, "id2label":ID2LABEL, "label2id":LABEL2ID}
    if force_float32: kwargs["dtype"] = torch.float32
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, **kwargs).to(DEVICE)
    loss_fn = nn.CrossEntropyLoss(
        weight=class_weights_tensor.to(DEVICE).to(model.dtype))
    no_decay = ["bias","LayerNorm.weight"]
    optimizer = torch.optim.AdamW([
        {"params":[p for n,p in model.named_parameters()
                   if not any(nd in n for nd in no_decay)],"weight_decay":0.01},
        {"params":[p for n,p in model.named_parameters()
                   if     any(nd in n for nd in no_decay)],"weight_decay":0.0},
    ], lr=lr)
    total_steps  = (len(tr_loader) // grad_accum) * num_epochs
    warmup_steps = int(0.1 * total_steps)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    best_macro_f1, best_state, patience_ctr = 0.0, None, 0
    history = {"train_loss":[],"val_loss":[],"val_macro_f1":[],"val_acc":[]}
    for epoch in range(num_epochs):
        t0 = time.time(); model.train(); total_loss = 0.0; optimizer.zero_grad()
        for step, batch in enumerate(tr_loader):
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            lbls = batch["labels"].to(DEVICE)
            kw   = {"input_ids":ids,"attention_mask":mask}
            if "token_type_ids" in batch:
                kw["token_type_ids"] = batch["token_type_ids"].to(DEVICE)
            loss = loss_fn(model(**kw).logits, lbls) / grad_accum
            loss.backward(); total_loss += loss.item() * grad_accum
            if (step+1) % grad_accum == 0:
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()
        _, val_acc, _, val_mf1, _, _, _, _ = evaluate(model, va_loader, loss_fn, DEVICE)
        avg_tr = total_loss / len(tr_loader)
        history["train_loss"].append(avg_tr)
        history["val_macro_f1"].append(val_mf1)
        history["val_acc"].append(val_acc)
        print(f"Epoch {epoch+1}/{num_epochs} ({time.time()-t0:.0f}s) | "
              f"TrainL={avg_tr:.4f} Acc={val_acc:.4f} MacroF1={val_mf1:.4f}")
        if val_mf1 > best_macro_f1:
            best_macro_f1 = val_mf1
            best_state = {k:v.clone() for k,v in model.state_dict().items()}
            patience_ctr = 0; print(f"  ✓ Best Macro F1: {best_macro_f1:.4f}")
        else:
            patience_ctr += 1
            if patience_ctr >= patience: print("  Early stopping."); break
    model.load_state_dict(best_state)
    _, val_acc, val_f1w, val_mf1, _, _, val_preds, val_true = \
        evaluate(model, va_loader, loss_fn, DEVICE)
    print(f"\nFINAL VAL — {model_name}")
    print(classification_report(val_true, val_preds,
                                 target_names=[ID2LABEL[i] for i in range(3)]))
    model.eval(); test_preds = []
    with torch.no_grad():
        for batch in te_loader:
            ids = batch["input_ids"].to(DEVICE)
            mask= batch["attention_mask"].to(DEVICE)
            kw  = {"input_ids":ids,"attention_mask":mask}
            if "token_type_ids" in batch:
                kw["token_type_ids"] = batch["token_type_ids"].to(DEVICE)
            test_preds.extend(torch.argmax(model(**kw).logits,dim=1).cpu().numpy())
    test_preds_labels = [ID2LABEL[p] for p in test_preds]
    metrics = {"model":model_name,"accuracy":round(val_acc,4),
               "macro_f1":round(val_mf1,4),"f1_weighted":round(val_f1w,4)}
    gc.collect(); torch.cuda.empty_cache()
    return metrics, val_preds, val_true, test_preds_labels, history

encoder_results    = {}
encoder_val_preds  = {}
encoder_val_true   = {}
encoder_test_preds = {}

for mname, ml, bs, ga, ep, lr, pat in [
    ("bert-base-uncased",       512, 16, 1, 8, 2e-5, 3),
    ("distilbert-base-uncased", 512, 16, 1, 8, 1e-5, 3),
]:
    m, vp, vt, tp, h = train_encoder(mname, ml, bs, ga, ep, lr, pat)
    key = mname.split("/")[-1]
    encoder_results[key]    = m
    encoder_val_preds[key]  = vp
    encoder_val_true[key]   = vt
    encoder_test_preds[key] = tp

print("\n=== BERT + DistilBERT complete ===")
for k,m in encoder_results.items():
    print(f"  {k:<30} Acc={m['accuracy']:.4f} Macro F1={m['macro_f1']:.4f}")

In [ ]:
# ============================================================
# CELL 4: Sanity Checks
# Verifies that key variables from Phase 2A are still in scope
# before proceeding to DeBERTa training (Phase 2B).
# Required because Kaggle may show stale variable states.
# ============================================================
print("encoder_results" in dir())
print("train_encoder" in dir())
print("val_split" in dir())

In [ ]:
# Additional check for train_df and DEVICE availability
print("train_df" in dir())
print("DEVICE" in dir())

In [ ]:
# ============================================================
# CELL 5: Encoder Baseline — DeBERTa-v3-base (Phase 2B)
#
# DeBERTa requires force_float32=True to avoid NaN loss
# when training with float16 (known issue with DeBERTa-v3).
# Uses longer training (20 epochs, patience=5) and gradient
# accumulation (ga=4) with smaller batch size (bs=4) due to
# larger model memory requirements.
#
# Settings: max_len=256, bs=4, ga=4, epochs=20, lr=2e-5, patience=5
# ============================================================
m, vp, vt, tp, h = train_encoder(
    "microsoft/deberta-v3-base", 256, 4, 4, 20, 2e-5, 5,
    force_float32=True)
key = "deberta-v3-base"
encoder_results[key]    = m
encoder_val_preds[key]  = vp
encoder_val_true[key]   = vt
encoder_test_preds[key] = tp
print(f"\nDeBERTa: Acc={m['accuracy']:.4f} Macro F1={m['macro_f1']:.4f}")

In [ ]:
# ============================================================
# CELL 6: Load Qwen3.5-0.8B (Phase 3)
#
# Qwen3.5-0.8B requires the transformers dev build (installed
# in Cell 1) because model_type 'qwen3_5' is not registered
# in stable releases. Key loading fix: _fast_init=False
# disables the buggy initialize_weights call in the dev build.
#
# Model is loaded in float16 to fit in T4 GPU VRAM (~16GB).
# VRAM usage after loading: ~0.78 GB out of 15.64 GB available.
# The model is used for all LLM-based inference: D3-Agentic,
# Zero-Shot baseline, DSPy optimization, and ablation study.
# ============================================================
import gc, re, os, random, time
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.metrics import (accuracy_score, f1_score,
                              precision_recall_fscore_support,
                              classification_report, confusion_matrix)

import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/huggingface/transformers.git",
    "--no-deps"
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "accelerate>=0.26.0"
])

# Force reload transformers
for mod in list(sys.modules.keys()):
    if "transformers" in mod:
        del sys.modules[mod]

from transformers import AutoTokenizer, AutoModelForCausalLM
import transformers
print(f"Transformers: {transformers.__version__}")

SEED   = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
print(f"Device: {DEVICE}")

D3_MODEL_NAME = "Qwen/Qwen3.5-0.8B"
print(f"\nLoading {D3_MODEL_NAME}...")
gc.collect(); torch.cuda.empty_cache()

d3_tokenizer = AutoTokenizer.from_pretrained(
    D3_MODEL_NAME, trust_remote_code=True, padding_side="left")
if d3_tokenizer.pad_token is None:
    d3_tokenizer.pad_token = d3_tokenizer.eos_token

d3_model = AutoModelForCausalLM.from_pretrained(
    D3_MODEL_NAME,
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    _fast_init=False          # ← disables the buggy initialize_weights call
)
d3_model.eval()

used  = torch.cuda.memory_allocated()/1e9
total = torch.cuda.get_device_properties(0).total_memory/1e9
print(f"VRAM: {used:.2f} / {total:.2f} GB")
print("Model loaded ✅")

In [ ]:
# ============================================================
# CELL 7: D3-Agentic Agent Definitions + Sanity Check (Phase 4)
#
# Implements the four-agent D3-Agentic Prompting framework:
#   Agent 1 (Question Intent): Identifies what the question
#     requires for a clear, direct answer.
#   Agent 2 (Answer Content): Extracts what the answer
#     actually covers, including omissions and vagueness.
#   Agent 3 (Gap & Evasion): Compares required vs actual
#     content to detect evasion, ambiguity, or partial replies.
#   Agent 4 (Decision): Assigns the final label using all
#     three prior analyses as context.
#
# Generation settings: do_sample=False (greedy), repetition
# penalty=1.1, max_length=3072. Think blocks are stripped
# from outputs via regex (Qwen3.5 sometimes emits <think> tags).
#
# Label parsing uses LABEL_VARIANTS dict for robust matching,
# checking the last output line first, then the full text.
# Falls back to "Ambivalent" if no valid label is found.
#
# A 2-example sanity check is run before full inference.
# ============================================================
VALID_LABELS   = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]
FALLBACK_LABEL = "Ambivalent"
LABEL_COL      = "label"
QUESTION_COL   = "question"
ANSWER_COL     = "answer"

LABEL_DEFS = (
    "- Clear Reply: The answer directly and fully addresses the question.\n"
    "- Ambivalent: The answer partially addresses the question, is vague, "
    "or deflects without fully committing.\n"
    "- Clear Non-Reply: The answer does not address the question at all, "
    "ignores it, or redirects entirely."
)

def generate(prompt_text, system_text, max_new_tokens=200):
    messages = [
        {"role": "system", "content": system_text},
        {"role": "user",   "content": prompt_text}
    ]
    try:
        fmt = d3_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=False)
    except TypeError:
        fmt = d3_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)

    inputs = d3_tokenizer(
        fmt, return_tensors="pt",
        truncation=True, max_length=3072).to(DEVICE)

    with torch.no_grad():
        out = d3_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=d3_tokenizer.pad_token_id,
            eos_token_id=d3_tokenizer.eos_token_id)

    raw = d3_tokenizer.decode(
        out[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True)
    raw = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL|re.IGNORECASE)
    raw = re.sub(r"<think>.*",          "", raw, flags=re.DOTALL|re.IGNORECASE)
    return raw.strip()

def agent1_question_intent(question):
    system = (
        "You are an expert analyst of political press conference discourse. "
        "Your task is to analyze what a journalist's question is asking for."
    )
    prompt = (
        f"Analyze this journalist's question from a political press conference.\n\n"
        f"Question: {question}\n\n"
        f"In 2-3 sentences, identify:\n"
        f"1. The core information the journalist is requesting.\n"
        f"2. What specific content a clear, direct answer would need to include.\n\n"
        f"Be concise and factual."
    )
    return generate(prompt, system, max_new_tokens=150)

def agent2_answer_content(answer):
    system = (
        "You are an expert analyst of political press conference discourse. "
        "Your task is to summarize what a politician's answer actually contains."
    )
    prompt = (
        f"Analyze this politician's answer from a press conference.\n\n"
        f"Answer: {answer}\n\n"
        f"In 2-3 sentences, describe:\n"
        f"1. What topics or claims the answer explicitly covers.\n"
        f"2. Any notable vagueness, deflections, or missing specifics.\n\n"
        f"Be concise and factual."
    )
    return generate(prompt, system, max_new_tokens=150)

def agent3_gap_evasion(question_intent, answer_content):
    system = (
        "You are an expert analyst of political press conference discourse. "
        "Your task is to identify gaps between what was asked and what was answered."
    )
    prompt = (
        f"You have analyzed a political press conference exchange.\n\n"
        f"What the question required:\n{question_intent}\n\n"
        f"What the answer actually contained:\n{answer_content}\n\n"
        f"In 2-3 sentences, assess:\n"
        f"1. Whether the answer covers what the question required.\n"
        f"2. Any gaps, evasions, partial responses, or ambiguities present.\n\n"
        f"Be concise and factual."
    )
    return generate(prompt, system, max_new_tokens=150)

def agent4_decision(question, answer, question_intent,
                    answer_content, gap_analysis):
    system = (
        "You are an expert classifier of political discourse. "
        "You assign response clarity labels based on structured analysis."
    )
    prompt = (
        f"You must classify the clarity of a politician's answer "
        f"to a journalist's question.\n\n"
        f"Label definitions:\n{LABEL_DEFS}\n\n"
        f"Original question: {question}\n\n"
        f"Original answer: {answer}\n\n"
        f"Analysis — What the question required:\n{question_intent}\n\n"
        f"Analysis — What the answer contained:\n{answer_content}\n\n"
        f"Analysis — Gaps and evasions detected:\n{gap_analysis}\n\n"
        f"Based on the analysis above, assign exactly one label.\n"
        f"Reply with ONLY one of these three options and nothing else:\n"
        f"Clear Reply | Ambivalent | Clear Non-Reply"
    )
    return generate(prompt, system, max_new_tokens=30)

LABEL_VARIANTS = {
    "Clear Reply": [
        "clear reply", "clearreply", "**clear reply**",
        "label: clear reply", "1. clear reply",
    ],
    "Ambivalent": [
        "ambivalent", "**ambivalent**", "label: ambivalent",
    ],
    "Clear Non-Reply": [
        "clear non-reply", "clear nonreply", "non-reply",
        "nonreply", "clear non reply", "label: clear non-reply",
        "label: clear non reply",
    ],
}

def parse_decision(text):
    if not text or not text.strip():
        return FALLBACK_LABEL, False
    lines  = [l.strip() for l in text.splitlines() if l.strip()]
    last   = lines[-1].lower() if lines else text.lower()
    for canonical, variants in LABEL_VARIANTS.items():
        for v in variants:
            if v in last:
                return canonical, True
    full_lower = text.lower()
    for canonical, variants in LABEL_VARIANTS.items():
        for v in variants:
            if v in full_lower:
                return canonical, True
    return FALLBACK_LABEL, False

def d3_classify(question, answer):
    q_intent     = agent1_question_intent(question)
    a_content    = agent2_answer_content(answer)
    gap          = agent3_gap_evasion(q_intent, a_content)
    raw_decision = agent4_decision(question, answer, q_intent, a_content, gap)
    label, valid = parse_decision(raw_decision)
    return {
        "agent1_output":   q_intent,
        "agent2_output":   a_content,
        "agent3_output":   gap,
        "agent4_raw":      raw_decision,
        "predicted_label": label,
        "is_valid":        valid,
    }

# ── Sanity check ──
print("=== Sanity Check (2 examples) ===\n")
set_seed()
for i in [0, 5]:
    row = val_split.iloc[i]
    print(f"--- Example {i} | True: {row[LABEL_COL]} ---")
    result = d3_classify(str(row[QUESTION_COL]), str(row[ANSWER_COL]))
    print(f"Agent 1: {result['agent1_output'][:100]}...")
    print(f"Agent 2: {result['agent2_output'][:100]}...")
    print(f"Agent 3: {result['agent3_output'][:100]}...")
    print(f"Agent 4: {result['agent4_raw']}")
    print(f"Predicted: {result['predicted_label']} (valid={result['is_valid']})\n")
print("Sanity check complete.")

In [ ]:
# ============================================================
# CELL 8: Full D3-Agentic Inference — Validation + Test (Phase 4)
#
# Runs the complete 4-agent D3 pipeline on all 690 validation
# examples and all 308 test examples.
#
# Validation results (~3.5 hrs):
#   Accuracy=0.5159, Macro F1=0.3373, Weighted F1=0.4812
#   Invalid outputs (no valid label parsed): 2/690
#   Key finding: model is heavily biased toward "Ambivalent";
#   Clear Non-Reply recall is near-zero (0.03).
#
# All agent outputs and predictions are saved to CSV files:
#   /kaggle/working/d3_val_results.csv
#   /kaggle/working/d3_test_results.csv
# ============================================================
print("Running D3-Agentic on validation set...")
set_seed()

d3_val_records = []
for _, row in tqdm(val_split.iterrows(),
                   total=len(val_split), desc="D3 val"):
    result = d3_classify(str(row[QUESTION_COL]), str(row[ANSWER_COL]))
    result["true_label"] = row[LABEL_COL]
    d3_val_records.append(result)

d3_val_df = val_split.copy().reset_index(drop=True)
for key in ["agent1_output","agent2_output","agent3_output",
            "agent4_raw","predicted_label","is_valid"]:
    d3_val_df[key] = [r[key] for r in d3_val_records]

y_true = d3_val_df[LABEL_COL].tolist()
y_pred = d3_val_df["predicted_label"].tolist()

d3_acc      = accuracy_score(y_true, y_pred)
d3_macro_f1 = f1_score(y_true, y_pred,
                        labels=VALID_LABELS, average="macro",
                        zero_division=0)
d3_f1w      = f1_score(y_true, y_pred,
                        labels=VALID_LABELS, average="weighted",
                        zero_division=0)
d3_invalid  = (~d3_val_df["is_valid"]).sum()

print(f"\n{'='*55}")
print(f"D3-AGENTIC (Qwen3.5-0.8B) — Validation Results")
print(f"{'='*55}")
print(f"  Accuracy  = {d3_acc:.4f}")
print(f"  Macro F1  = {d3_macro_f1:.4f}")
print(f"  Weighted F1 = {d3_f1w:.4f}")
print(f"  Invalid outputs = {d3_invalid}/{len(val_split)}")
print()
print(classification_report(y_true, y_pred,
                              labels=VALID_LABELS,
                              zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=VALID_LABELS)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=VALID_LABELS, yticklabels=VALID_LABELS, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion Matrix — D3-Agentic Qwen3.5-0.8B", fontweight="bold")
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix_d3.png",
            dpi=150, bbox_inches="tight")
plt.show()

print("\nRunning D3-Agentic on test set...")
set_seed()

d3_test_records = []
for _, row in tqdm(test_df.iterrows(),
                   total=len(test_df), desc="D3 test"):
    result = d3_classify(str(row[QUESTION_COL]), str(row[ANSWER_COL]))
    d3_test_records.append(result)

d3_test_df = test_df.copy().reset_index(drop=True)
for key in ["agent1_output","agent2_output","agent3_output",
            "agent4_raw","predicted_label","is_valid"]:
    d3_test_df[key] = [r[key] for r in d3_test_records]

print(f"Test invalid outputs: "
      f"{(~d3_test_df['is_valid']).sum()}/{len(test_df)}")
print("\nTest prediction distribution:")
print(d3_test_df["predicted_label"].value_counts())

d3_val_df.to_csv("/kaggle/working/d3_val_results.csv", index=False)
d3_test_df.to_csv("/kaggle/working/d3_test_results.csv", index=False)
print("\nResults saved.")
print("\n--- Phase 3+4 Complete ---")

In [ ]:
# ============================================================
# CELL 9: Submission File + Full System Comparison (Phase 5)
#
# Stores D3-Agentic results in encoder_results dict for unified
# comparison. Generates a bar chart comparing all 4 systems
# on Macro F1 and Accuracy (saved as full_comparison.png).
#
# Best system: DeBERTa-v3-base (Macro F1=0.6345 on val).
# The fine-tuned encoder outperforms the zero-shot LLM approach
# because the task requires subtle linguistic distinctions that
# benefit from gradient-based task adaptation.
#
# Two submission files are saved:
#   submission_best_d3_agentic_system.csv — DeBERTa predictions
#     (best overall system, submitted to Kaggle competition)
#   submission_d3_agentic.csv — D3-Agentic predictions
#     (for reference and comparison in the report)
# ============================================================
encoder_results["d3_agentic"] = {
    "model":       "D3-Agentic Qwen3.5-0.8B",
    "accuracy":    0.5159,
    "macro_f1":    0.3373,
    "f1_weighted": 0.4812,
}

print("\n" + "="*65)
print("FULL SYSTEM COMPARISON — Validation Set")
print("="*65)
print(f"{'Model':<35} {'Accuracy':>9} {'Macro F1':>9}")
print("─"*55)
for key, m in encoder_results.items():
    print(f"  {m['model']:<33} {m['accuracy']:>9.4f} {m['macro_f1']:>9.4f}")
print("─"*55)
print("Best overall: deberta-v3-base (Macro F1=0.6344)")
print("="*65)

names  = [m["model"].split("/")[-1] for m in encoder_results.values()]
macros = [m["macro_f1"] for m in encoder_results.values()]
accs   = [m["accuracy"] for m in encoder_results.values()]

colors_mf1 = ["#4C72B0","#4C72B0","#4C72B0","#DD4444"]
colors_acc = ["#DD8452","#DD8452","#DD8452","#FF8888"]

x = np.arange(len(names)); width = 0.35
fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - width/2, macros, width, label="Macro F1", color=colors_mf1)
bars2 = ax.bar(x + width/2, accs,   width, label="Accuracy", color=colors_acc)
for bar, v in [(b,v) for bars,vals in [(bars1,macros),(bars2,accs)]
               for b,v in zip(bars,vals)]:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f"{v:.3f}", ha="center", fontsize=8)
ax.set_title("All Systems — Macro F1 vs Accuracy (Val)", fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(names, rotation=15, ha="right")
ax.set_ylim(0, 0.80); ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("/kaggle/working/full_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

d3_test_df = pd.read_csv("/kaggle/working/d3_test_results.csv")

submission = pd.DataFrame({
    "Id":        test_df["id"].values,
    "Predicted": encoder_test_preds["deberta-v3-base"]
})
submission.to_csv("/kaggle/working/submission_best_d3_agentic_system.csv", index=False)
print(f"\nSubmission saved: {len(submission)} rows")
print(submission["Predicted"].value_counts())
print("\nSample:")
print(submission.head(10))

d3_submission = pd.DataFrame({
    "Id":        test_df["id"].values,
    "Predicted": d3_test_df["predicted_label"].values
})
d3_submission.to_csv("/kaggle/working/submission_d3_agentic.csv", index=False)
print(f"\nD3 submission saved.")
print(d3_submission["Predicted"].value_counts())

In [ ]:
# ============================================================
# CELL 10: Zero-Shot Qwen3.5-0.8B Baseline (HW3 Reproduction)
#
# Implements a single-prompt zero-shot classifier using the
# same Qwen3.5-0.8B model as D3-Agentic, but with no agentic
# decomposition — just one system prompt + question/answer pair.
#
# This serves as the HW3 baseline and directly isolates the
# contribution of the multi-agent pipeline: comparing Zero-Shot
# vs D3-Agentic with the same model shows whether decomposing
# into 4 agents actually helps classification.
#
# Key result: Zero-Shot Macro F1=0.173 vs D3-Agentic F1=0.337
# Zero-Shot collapses to predicting "Clear Non-Reply" for
# nearly all test examples, confirming that without structured
# decomposition the small LLM cannot solve this task reliably.
#
# Outputs saved:
#   /kaggle/working/zeroshot_test_results.csv
#   /kaggle/working/submission_zeroshot_qwen.csv
# ============================================================
import re, time
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report

ZEROSHOT_SYSTEM = """You are an expert political analyst specializing in evasion detection.
Your task is to classify political answers into exactly one of three categories:

- Clear Reply: The answer directly and substantively addresses the question asked.
- Ambivalent: The answer partially addresses the question but remains vague, hedges, or only partly responds.
- Clear Non-Reply: The answer completely avoids, deflects, or refuses to address the question.

Respond with ONLY one of these exact labels: Clear Reply, Ambivalent, Clear Non-Reply"""

def zeroshot_classify(question, answer, max_new_tokens=20):
    prompt = f"""Question: {question[:400]}

Answer: {answer[:600]}

Classification:"""
    messages = [
        {"role": "system", "content": ZEROSHOT_SYSTEM},
        {"role": "user", "content": prompt}
    ]
    text = d3_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = d3_tokenizer(text, return_tensors="pt").to(d3_model.device)
    with torch.no_grad():
        out = d3_model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            repetition_penalty=1.1, pad_token_id=d3_tokenizer.eos_token_id
        )
    generated = d3_tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()
    generated = re.sub(r"<think>.*?</think>", "", generated, flags=re.DOTALL).strip()
    for label in ["Clear Non-Reply", "Clear Reply", "Ambivalent"]:
        if label.lower() in generated.lower():
            return generated, label, True
    return generated, "Ambivalent", False

print("Running Zero-Shot on validation set...")
zs_val_preds, zs_val_raw, zs_val_valid = [], [], []
t0 = time.time()
for i, row in val_split.iterrows():
    raw, pred, valid = zeroshot_classify(row["question"], row["answer"])
    zs_val_preds.append(pred)
    zs_val_raw.append(raw)
    zs_val_valid.append(valid)
    if len(zs_val_preds) % 100 == 0:
        print(f"  {len(zs_val_preds)}/690 done ({time.time()-t0:.0f}s)")

zs_val_acc = accuracy_score(val_split["label"], zs_val_preds)
zs_val_mf1 = f1_score(val_split["label"], zs_val_preds, average="macro")
zs_val_wf1 = f1_score(val_split["label"], zs_val_preds, average="weighted")
zs_invalid = sum(1 for v in zs_val_valid if not v)

print(f"\n{'='*55}")
print("ZERO-SHOT Qwen3.5-0.8B — Validation Results")
print(f"{'='*55}")
print(f"  Accuracy   = {zs_val_acc:.4f}")
print(f"  Macro F1   = {zs_val_mf1:.4f}")
print(f"  Weighted F1= {zs_val_wf1:.4f}")
print(f"  Invalid    = {zs_invalid}/690")
print()
print(classification_report(val_split["label"], zs_val_preds))

encoder_results["zeroshot_qwen"] = {
    "model":       "Zero-Shot Qwen3.5-0.8B",
    "accuracy":    zs_val_acc,
    "macro_f1":    zs_val_mf1,
    "f1_weighted": zs_val_wf1,
}

print("Running Zero-Shot on test set...")
zs_test_preds, zs_test_raw = [], []
for i, row in test_df.iterrows():
    raw, pred, _ = zeroshot_classify(row["question"], row["answer"])
    zs_test_preds.append(pred)
    zs_test_raw.append(raw)
    if len(zs_test_preds) % 100 == 0:
        print(f"  {len(zs_test_preds)}/308 done")

zs_test_df = pd.DataFrame({
    "Id": test_df["id"].values,
    "Predicted": zs_test_preds,
    "raw_output": zs_test_raw
})
zs_test_df.to_csv("/kaggle/working/zeroshot_test_results.csv", index=False)

zs_submission = pd.DataFrame({
    "Id": test_df["id"].values,
    "Predicted": zs_test_preds
})
zs_submission.to_csv("/kaggle/working/submission_zeroshot_qwen.csv", index=False)
print(f"\nZero-shot test distribution:\n{zs_submission['Predicted'].value_counts()}")
print("Zero-shot baseline complete.")

In [ ]:
# ============================================================
# CELL 11: Prompt Optimization with DSPy BootstrapFewShot
#
# Uses DSPy's BootstrapFewShot optimizer to automatically
# select few-shot demonstrations for the D3 classification
# pipeline. This directly addresses the prompt optimization
# requirement from the assignment guidelines.
#
# A custom QwenLM wrapper adapts Qwen3.5-0.8B to the DSPy
# LM interface. The QuestionIntent signature defines the
# classification task as a structured DSPy program.
#
# Optimization runs on 12 training examples (limited for
# speed); evaluation is performed on 20 held-out val examples.
# The optimized system selects demonstrations that maximize
# the exact-match label accuracy metric.
# ============================================================
!pip install dspy-ai -q

import dspy
from dspy.teleprompt import BootstrapFewShot

class QwenLM(dspy.LM):
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.kwargs = {"temperature": 0.0, "max_tokens": 200}
    
    def basic_request(self, prompt, **kwargs):
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            out = self.model.generate(
                **inputs, max_new_tokens=200, do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id
            )
        text = self.tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        return {"choices": [{"message": {"content": text}}]}
    
    def __call__(self, prompt=None, messages=None, **kwargs):
        if messages:
            prompt = "\n".join(m["content"] for m in messages)
        return [self.basic_request(prompt)["choices"][0]["message"]["content"]]

class QuestionIntent(dspy.Signature):
    """Analyze a political interview question for response-clarity classification."""
    question = dspy.InputField(desc="The interview question")
    answer   = dspy.InputField(desc="The politician's answer")
    intent_analysis = dspy.OutputField(desc="What information a clear answer must provide")
    coverage_gaps   = dspy.OutputField(desc="What gaps exist between question needs and answer")
    label = dspy.OutputField(desc="One of: Clear Reply, Ambivalent, Clear Non-Reply")

class D3Classifier(dspy.Module):
    def __init__(self):
        super().__init__()
        self.analyze = dspy.ChainOfThought(QuestionIntent)
    
    def forward(self, question, answer):
        return self.analyze(question=question, answer=answer)

def make_dspy_examples(df, n=50):
    examples = []
    for _, row in df.sample(n=n, random_state=42).iterrows():
        ex = dspy.Example(
            question=row["question"][:400],
            answer=row["answer"][:600],
            label=row["label"]
        ).with_inputs("question", "answer")
        examples.append(ex)
    return examples

def dspy_metric(example, pred, trace=None):
    return pred.label.strip() == example.label

lm = QwenLM(d3_model, d3_tokenizer)
dspy.settings.configure(lm=lm)

train_examples = make_dspy_examples(train_df, n=50)
val_examples   = make_dspy_examples(val_split, n=20)

optimizer = BootstrapFewShot(metric=dspy_metric, max_bootstrapped_demos=3, max_rounds=1)
student = D3Classifier()

print("Running DSPy BootstrapFewShot optimization...")
optimized = optimizer.compile(student, trainset=train_examples[:12])
print("Optimization complete.")

print("\nEvaluating optimized DSPy system on 20 val examples...")
dspy_preds = []
for ex in val_examples:
    try:
        pred = optimized(question=ex.question, answer=ex.answer)
        raw = pred.label.strip()
        for lbl in ["Clear Non-Reply", "Clear Reply", "Ambivalent"]:
            if lbl.lower() in raw.lower():
                dspy_preds.append(lbl); break
        else:
            dspy_preds.append("Ambivalent")
    except:
        dspy_preds.append("Ambivalent")

true_subset = [ex.label for ex in val_examples]
dspy_mf1 = f1_score(true_subset, dspy_preds, average="macro", zero_division=0)
print(f"DSPy Optimized Macro F1 (20 val): {dspy_mf1:.4f}")
print(classification_report(true_subset, dspy_preds, zero_division=0))

In [ ]:
# ============================================================
# CELL 12: Agent Ablation Study
#
# Systematically removes individual agents from the D3 pipeline
# to quantify each agent's contribution to classification.
# Runs 5 configurations on 100 sampled validation examples:
#
#   Full D3 (all 4 agents)         — complete pipeline
#   No Agent1 (skip question intent) — removes intent analysis
#   No Agent2 (skip answer content)  — removes content analysis
#   No Agent3 (skip gap analysis)    — removes evasion detection
#   Decision only (no agents)        — direct classification
#
# The Decision Agent always runs; only Agents 1-3 are toggled.
# Results answer the assignment question: "Which parts of the
# pipeline are necessary? Does removing them reduce performance?"
# ============================================================
import re

def d3_ablation(question, answer, skip_agents=None, max_new_tokens=30):
    skip_agents = skip_agents or []
    
    def call_agent(system_prompt, user_content, max_tok=180):
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_content}
        ]
        text = d3_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
        inputs = d3_tokenizer(text, return_tensors="pt").to(d3_model.device)
        with torch.no_grad():
            out = d3_model.generate(
                **inputs, max_new_tokens=max_tok, do_sample=False,
                repetition_penalty=1.1, pad_token_id=d3_tokenizer.eos_token_id
            )
        resp = d3_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        return re.sub(r"<think>.*?</think>", "", resp, flags=re.DOTALL).strip()
    
    a1 = call_agent("Analyze what information the question requires for a clear answer.",
                    f"Question: {question[:400]}") if 1 not in skip_agents else "N/A"
    
    a2 = call_agent("Summarize what the answer actually addresses.",
                    f"Answer: {answer[:600]}") if 2 not in skip_agents else "N/A"
    
    a3_input = f"Question needs: {a1[:200]}\nAnswer provides: {a2[:200]}"
    a3 = call_agent("Identify gaps between what was asked and what was answered.",
                    a3_input) if 3 not in skip_agents else "N/A"
    
    # Decision agent always runs
    decision_input = f"Q: {question[:300]}\nA: {answer[:400]}\n"
    if 1 not in skip_agents: decision_input += f"\nAgent1: {a1[:150]}"
    if 2 not in skip_agents: decision_input += f"\nAgent2: {a2[:150]}"
    if 3 not in skip_agents: decision_input += f"\nAgent3: {a3[:150]}"
    decision_input += "\n\nClassify: Clear Reply, Ambivalent, or Clear Non-Reply?"
    
    raw = call_agent(
        "You are a political evasion classifier. Output exactly one label.",
        decision_input, max_tok=30
    )
    for lbl in ["Clear Non-Reply", "Clear Reply", "Ambivalent"]:
        if lbl.lower() in raw.lower():
            return raw, lbl
    return raw, "Ambivalent"

ablation_subset = val_split.sample(n=100, random_state=42).reset_index(drop=True)
true_abl = ablation_subset["label"].tolist()

ablation_configs = {
    "Full D3 (all 4 agents)": [],
    "No Agent1 (skip question intent)": [1],
    "No Agent2 (skip answer content)": [2],
    "No Agent3 (skip gap analysis)": [3],
    "Decision only (no agents)": [1, 2, 3],
}

ablation_results = {}
for config_name, skip in ablation_configs.items():
    print(f"\nRunning: {config_name}...")
    preds = []
    for _, row in ablation_subset.iterrows():
        _, pred = d3_ablation(row["question"], row["answer"], skip_agents=skip)
        preds.append(pred)
    mf1 = f1_score(true_abl, preds, average="macro")
    acc  = accuracy_score(true_abl, preds)
    ablation_results[config_name] = {"macro_f1": mf1, "accuracy": acc, "preds": preds}
    print(f"  Macro F1={mf1:.4f}  Acc={acc:.4f}")

print("\n" + "="*60)
print("ABLATION RESULTS (100 val examples)")
print("="*60)
for name, res in ablation_results.items():
    print(f"  {name:<40} Macro F1={res['macro_f1']:.4f}")

In [ ]:
# ============================================================
# CELL 13: Error Analysis + Final System Comparison
#
# Produces three key outputs for the report:
#
# 1. Confusion matrices (5 systems side by side):
#    Shows per-class prediction patterns. Key finding:
#    Zero-Shot collapses to Clear Non-Reply; D3-Agentic
#    collapses to Ambivalent; encoders are more balanced.
#
# 2. Per-class F1 bar chart:
#    Directly compares how each system handles each class.
#    Clear Non-Reply is hardest for all prompting approaches.
#
# 3. Textual error analysis:
#    - Per-class breakdown of misclassifications for D3 and DeBERTa
#    - Answer length vs D3 accuracy (does longer answer = easier?)
#    - Final summary table: Accuracy, Macro F1, per-class F1
#
# Note: encoder val preds are stored as integer IDs;
# to_labels() converts them to string labels before metric
# computation to avoid sklearn "mix of types" ValueError.
# ============================================================
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

CLASSES = ["Ambivalent", "Clear Non-Reply", "Clear Reply"]

# Convert integer preds to label strings
def to_labels(preds):
    if preds and isinstance(preds[0], (int, np.integer)):
        return [ID2LABEL[p] for p in preds]
    return list(preds)

systems = {
    "BERT":       to_labels(encoder_val_preds["bert-base-uncased"]),
    "DistilBERT": to_labels(encoder_val_preds["distilbert-base-uncased"]),
    "DeBERTa":    to_labels(encoder_val_preds["deberta-v3-base"]),
    "Zero-Shot":  zs_val_preds,
    "D3-Agentic": pd.read_csv("/kaggle/working/d3_val_results.csv")["predicted_label"].tolist(),
}
true_labels = val_split["label"].tolist()

# ── 1. Confusion matrices ──
fig, axes = plt.subplots(1, 5, figsize=(30, 5))
for ax, (name, preds) in zip(axes, systems.items()):
    cm = confusion_matrix(true_labels, preds, labels=CLASSES)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Amb", "CNR", "CR"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    mf1 = f1_score(true_labels, preds, average="macro", zero_division=0)
    ax.set_title(f"{name}\nMacro F1={mf1:.3f}", fontsize=10)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.suptitle("Confusion Matrices — Validation Set", fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

# ── 2. Per-class F1 bar chart ──
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(CLASSES))
w = 0.15
colors = ["#4C72B0","#DD8452","#55A868","#C44E52","#8172B3"]
for i, (name, preds) in enumerate(systems.items()):
    f1s = f1_score(true_labels, preds, average=None,
                   labels=CLASSES, zero_division=0)
    ax.bar(x + i*w, f1s, w, label=name, color=colors[i])
    for j, v in enumerate(f1s):
        ax.text(x[j] + i*w, v + 0.005, f"{v:.2f}", ha="center", fontsize=7)
ax.set_xticks(x + 2*w)
ax.set_xticklabels(CLASSES)
ax.set_ylim(0, 0.95)
ax.set_ylabel("F1 Score")
ax.set_title("Per-Class F1 by System — Validation Set", fontweight="bold")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("/kaggle/working/per_class_f1.png", dpi=150, bbox_inches="tight")
plt.show()

# ── 3. Error analysis ──
d3_val_analysis = pd.read_csv("/kaggle/working/d3_val_results.csv")
val_copy = val_split.copy().reset_index(drop=True)
val_copy["d3_pred"]      = d3_val_analysis["predicted_label"].values
val_copy["deberta_pred"] = to_labels(encoder_val_preds["deberta-v3-base"])
val_copy["zs_pred"]      = zs_val_preds
val_copy["a_len"]        = val_copy["answer"].str.len()

print("\n=== D3-Agentic Error Analysis ===")
for cls in CLASSES:
    subset  = val_copy[val_copy["label"] == cls]
    correct = (subset["d3_pred"] == cls).sum()
    total   = len(subset)
    wrong   = subset[subset["d3_pred"] != cls]["d3_pred"].value_counts().to_dict()
    print(f"  {cls}: {correct}/{total} correct | Misclassified as: {wrong}")

print("\n=== DeBERTa Error Analysis ===")
for cls in CLASSES:
    subset  = val_copy[val_copy["label"] == cls]
    correct = (subset["deberta_pred"] == cls).sum()
    total   = len(subset)
    wrong   = subset[subset["deberta_pred"] != cls]["deberta_pred"].value_counts().to_dict()
    print(f"  {cls}: {correct}/{total} correct | Misclassified as: {wrong}")

print("\n=== Answer Length vs D3 Accuracy ===")
val_copy["d3_correct"] = val_copy["d3_pred"] == val_copy["label"]
val_copy["a_len_bin"]  = pd.qcut(val_copy["a_len"], q=4,
                                  labels=["Short","Med-Short","Med-Long","Long"])
print(val_copy.groupby("a_len_bin", observed=True)["d3_correct"].agg(["mean","count"]).round(3))

# ── 4. Final summary ──
print("\n" + "="*70)
print("FINAL SYSTEM COMPARISON — Val Set")
print("="*70)
print(f"{'System':<22} {'Acc':>7} {'MacroF1':>9} {'Amb F1':>8} {'CNR F1':>8} {'CR F1':>8}")
print("-"*70)
for name, preds in systems.items():
    acc = accuracy_score(true_labels, preds)
    mf1 = f1_score(true_labels, preds, average="macro", zero_division=0)
    f1s = f1_score(true_labels, preds, average=None,
                   labels=CLASSES, zero_division=0)
    print(f"  {name:<20} {acc:>7.4f} {mf1:>9.4f} {f1s[0]:>8.3f} {f1s[1]:>8.3f} {f1s[2]:>8.3f}")
print("="*70)
print("Files: confusion_matrices.png, per_class_f1.png")